# **KLASIFIKASI BERITA MENGGUNAKAN LATENT DIRICHLET ALLOCATION (LDA)**

## Load hasil preprocessing

In [1]:
import pandas as pd

df_tempo_processed_csv = pd.read_csv('/content/tempo_preprocessed.csv')
display(df_tempo_processed_csv)

,id_berita,judul_berita,isi_berita,kategori_berita,text,clean_text,tokens
0,2076486,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,TENTARA Nasional Indonesia atau TNI menggelar ...,politik,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"['jenderal', 'defile', 'hut', 'tni', 'tentara'..."
1,2076480,Prabowo Minta Semua Pesantren Didata setelah P...,PRESIDENPrabowoSubianto memerintahkan semua po...,politik,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"['prabowo', 'pesantren', 'didata', 'ponpes', '..."
2,2076479,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,PRESIDEN Prabowo Subianto memerintahkan Pangli...,politik,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"['prabowo', 'utamakan', 'kompetensi', 'prajuri..."
3,2076473,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,KEMENTERIAN Komunikasi dan Digital (Kemenkomdi...,politik,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"['kemenkomdigi', 'permintaan', 'data', 'tiktok..."
4,2076468,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,MANTAN Presiden Megawati Soekarnoputri dan Jok...,politik,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"['megawati', 'jokowi', 'hadir', 'hut', 'tni', ..."
...,...,...,...,...,...,...,...
895,2073886,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ADA dua kondisi yang kini melekat padaHarry Ka...,sepakbola,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"['harry', 'kane', 'memecahkan', 'rekor', 'gol'..."
896,2073880,Peluang Timnas Indonesia Lewati Hadangan Arab ...,PENGAMAT sepak bola Tanah Air Kesit Budi Hando...,sepakbola,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"['peluang', 'timnas', 'indonesia', 'lewati', '..."
897,2073852,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"DALAM usia 40 tahun,Cristiano Ronaldomasih mam...",sepakbola,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"['ketajaman', 'cristiano', 'ronaldo', 'nassr',..."
898,2073819,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"BADAN sepak bola dunia,FIFA, menjatuhkan sanks...",sepakbola,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"['fifa', 'jatuhkan', 'sanksi', 'malaysia', 'pe..."


## Siapkan teks untuk LDA

In [3]:
df_tempo_processed_csv['tokens_list'] = df_tempo_processed_csv['tokens'].apply(eval)
df_tempo_processed_csv['tokens_str'] = df_tempo_processed_csv['tokens_list'].apply(lambda x: " ".join(x))

## Buat representasi “bag of words”

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df_tempo_processed_csv['tokens_str'])

## Jalankan LDA untuk menemukan topik

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
X_topics = lda_model.fit_transform(X_bow)


## Lihat Hasik Topik

In [7]:
words = vectorizer.get_feature_names_out()
for i, topic in enumerate(lda_model.components_):
    print(f"Topik {i+1}:")
    print([words[i] for i in topic.argsort()[:-10 - 1:-1]])
    print()


Topik 1:
['tiktok', 'data', 'oktober', 'digital', 'komdigi', 'pemerintah', 'akun', 'sistem', 'kementerian', 'pilihan']

Topik 2:
['pertandingan', 'gol', 'pemain', 'liga', 'laga', 'babak', 'kemenangan', 'tim', 'menit', 'bermain']

Topik 3:
['indonesia', 'pemain', 'games', 'sea', 'timnas', 'jakarta', 'indra', 'keluarga', 'oktober', 'sampah']

Topik 4:
['penerbangan', 'wisata', 'harga', 'iran', 'dunia', 'gram', 'kredit', 'emas', 'tiket', 'veto']

Topik 5:
['israel', 'gaza', 'trump', 'palestina', 'negara', 'anggota', 'hukum', 'dpr', 'kpk', 'korupsi']

Topik 6:
['gempa', 'gunung', 'kali', 'september', 'rumah', 'bmkg', 'aktivitas', 'kilometer', 'janice', 'pesisir']

Topik 7:
['indonesia', 'persen', 'oktober', 'triliun', 'september', 'saham', 'pasar', 'keuangan', 'laut', 'pemerintah']

Topik 8:
['kapal', 'jakarta', 'oktober', 'orang', 'hujan', 'korban', 'wilayah', 'pilihan', 'global', 'sumud']

Topik 9:
['tni', 'oktober', 'prabowo', 'hut', 'jakarta', 'jalan', 'indonesia', 'presiden', 'tanah',

## Siapkan data klasifikasi

In [11]:
from sklearn.model_selection import train_test_split

# Gunakan dataframe yang benar
X = X_topics
y = df_tempo_processed_csv["kategori_berita"]

# Split data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Lakukan Klasifikasi

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


               precision    recall  f1-score   support

      ekonomi       0.41      0.32      0.36        22
      hiburan       0.50      0.27      0.35        22
        hukum       0.22      0.38      0.28        13
internasional       0.75      0.26      0.39        23
   lingkungan       0.41      0.64      0.50        22
     olahraga       0.33      0.28      0.30        18
     otomotif       0.70      0.88      0.78        16
      politik       0.28      0.32      0.30        25
    sepakbola       0.64      0.74      0.68        19

     accuracy                           0.44       180
    macro avg       0.47      0.45      0.44       180
 weighted avg       0.47      0.44      0.43       180

